# **Detect Faces from our previous dataset**

## Why Filter Images for Background Complexity?
GAN models are highly sensitive to **irrelevant background details**. Images with cluttered or complex backgrounds can confuse the generator, causing it to:

- Misrepresent the main subject
- Produce artifacts around edges
- Struggle to learn consistent style or color patterns

To address this, we perform **background filtering**:

1. **Person Detection**  
   - Using YOLOv8, we keep images that contain **exactly one person**, ensuring the model focuses on a single subject.

2. **Background Simplicity Check**  
   - Using edge detection, we filter out images with too many edges or background clutter.
   - Only images with a “simple” background (below a threshold of edge intensity) are retained.

This preprocessing ensures that the dataset has **clean, centered subjects**, which improves GAN training stability and produces higher-quality generated images.

In [3]:
pip install ultralytics

In [ ]:
# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import zipfile
import os

zip_path = "/content/drive/MyDrive/GAN/dataset_GAN.zip"
extract_path = "/content/dataset_GAN_2"

# Extract the images from the zip file
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset descomprimido")

Dataset descomprimido


In [ ]:
from ultralytics import YOLO
import shutil
import cv2
import numpy as np

# Dowload yolo8 model
model = YOLO("yolov8n.pt")

def is_simple_background(image_path, threshold=100):
    img = cv2.imread(image_path, 0)
    edges = cv2.Canny(img, 100, 200)
    return edges.mean() < threshold

def filter_folder(input_dir, output_dir):
    os.makedirs(output_dir, exist_ok=True)

    for img in os.listdir(input_dir):
        path = os.path.join(input_dir, img)

        try:
            results = model(path)[0]
            persons = [box for box in results.boxes if int(box.cls) == 0]

            if len(persons) == 1 and is_simple_background(path):
                shutil.copy(path, os.path.join(output_dir, img))

        except:
            pass

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [7]:
# TRAIN
filter_folder(
    "/content/dataset_GAN_2/train/cartoon",
    "/content/dataset_GAN_2/filtered/train/cartoon"
)

filter_folder(
    "/content/dataset_GAN_2/train/live_action",
    "/content/dataset_GAN_2/filtered/train/live_action"
)

# TEST
filter_folder(
    "/content/dataset_GAN_2/test/cartoon",
    "/content/dataset_GAN_2/filtered/test/cartoon"
)
filter_folder(
    "/content/dataset_GAN_2/test/live_action",
    "/content/dataset_GAN_2/filtered/test/live_action"
)

Se han truncado las últimas 5000 líneas del flujo de salida.
image 1/1 /content/dataset_GAN_2/train/live_action/BellaBestia2_LA_00501.jpg: 640x640 2 persons, 341.1ms
Speed: 11.5ms preprocess, 341.1ms inference, 4.0ms postprocess per image at shape (1, 3, 640, 640)

image 1/1 /content/dataset_GAN_2/train/live_action/BellaBestia2_LA_00112.jpg: 640x640 1 person, 439.6ms
Speed: 26.3ms preprocess, 439.6ms inference, 5.2ms postprocess per image at shape (1, 3, 640, 640)

image 1/1 /content/dataset_GAN_2/train/live_action/Aurora_LA_00189.jpg: 640x640 1 person, 438.9ms
Speed: 9.4ms preprocess, 438.9ms inference, 2.4ms postprocess per image at shape (1, 3, 640, 640)

image 1/1 /content/dataset_GAN_2/train/live_action/Blancanieves_LA_00178.jpg: 640x640 2 persons, 427.7ms
Speed: 19.0ms preprocess, 427.7ms inference, 2.0ms postprocess per image at shape (1, 3, 640, 640)

image 1/1 /content/dataset_GAN_2/train/live_action/Aladdin_LA_00138.jpg: 640x640 2 persons, 1 tie, 468.6ms
Speed: 10.5ms preproc

In [9]:
def count_images(path):
    return len([f for f in os.listdir(path) if f.lower().endswith(('.png','.jpg','.jpeg'))])

print("TRAIN CARTOON:", count_images("/content/dataset_GAN_2/filtered/train/cartoon"))
print("TEST CARTOON:", count_images("/content/dataset_GAN_2/filtered/test/cartoon"))
print("TRAIN LA:", count_images("/content/dataset_GAN_2/filtered/train/live_action"))
print("TEST LA:", count_images("/content/dataset_GAN_2/filtered/test/live_action"))

TRAIN CARTOON: 567
TEST CARTOON: 55
TRAIN LA: 586
TEST LA: 56


In [ ]:
# Save our new datasets
shutil.make_archive("/content/drive/MyDrive/GAN/dataset_GAN_faces",'zip',"/content/dataset_GAN_2/filtered")
print("Zip guardado en Drive")

Zip guardado en Drive
